<a href="https://colab.research.google.com/github/mohadeseh-mansoorian/-9/blob/main/jks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
!pip install -q requests pandas openpyxl pyjks cryptography beautifulsoup4

import os
import re
import time
import requests
import pandas as pd
import jks
import warnings
from cryptography import x509
from cryptography.hazmat.primitives import hashes
from urllib.parse import quote
from datetime import datetime, timezone
from bs4 import BeautifulSoup
from IPython.display import display
from google.colab import files

warnings.filterwarnings("ignore")
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})
print("آماده شروع هستیم.")

آماده شروع هستیم.


In [9]:
API = "https://api.github.com"
candidates = []

# ۱. گیت‌هاب (منطق دقیقاً مثل فایل خودتان)
print("جستجو در GitHub...")
queries = ["jks java", "java keystore", "jks test"]
for query in queries:
    try:
        res = session.get(f"{API}/search/repositories", params={"q": query, "per_page": 5}, timeout=25)
        if res.status_code == 200:
            for repo in res.json().get("items", []):
                repo_name, branch, repo_url = repo["full_name"], repo["default_branch"], repo["html_url"]
                tree = session.get(f"{API}/repos/{repo_name}/git/trees/{quote(branch, safe='')}?recursive=1", timeout=30)
                if tree.status_code == 200:
                    for item in tree.json().get("tree", []):
                        if item.get("type") == "blob" and item["path"].lower().endswith((".jks", ".keystore")) and item.get("size", 0) <= 2000000:
                            candidates.append({
                                "Platform": "GitHub", "Repository": repo_name, "Branch": branch, "Path": item["path"],
                                "Repository URL": repo_url,
                                "File URL": f"https://github.com/{repo_name}/blob/{quote(branch, safe='')}/{quote(item['path'], safe='/')}"
                            })
    except Exception: pass

# ۲. گیت‌لب (GitLab API)
print("جستجو در GitLab...")
try:
    res = session.get("https://gitlab.com/api/v4/projects", params={"search": "jks", "per_page": 5})
    for proj in res.json():
        candidates.append({"Platform": "GitLab", "Repository": proj["path_with_namespace"], "Branch": "main", "Path": "unknown.jks", "Repository URL": proj["web_url"], "File URL": "pending"})
except Exception: pass

# ۳. بیت‌باکت (Bitbucket API)
print("جستجو در Bitbucket...")
try:
    res = session.get("https://api.bitbucket.org/2.0/repositories", params={"q": "name~jks", "pagelen": 5})
    for repo in res.json().get("values", []):
        candidates.append({"Platform": "Bitbucket", "Repository": repo["full_name"], "Branch": "main", "Path": "unknown.jks", "Repository URL": repo["links"]["html"]["href"], "File URL": "pending"})
except Exception: pass

# ۴. استک‌اورفلو (StackOverflow API - پیدا کردن لینک‌های مستقیم JKS)
print("جستجو در StackOverflow...")
try:
    res = session.get("https://api.stackexchange.com/2.3/search/advanced", params={"q": "jks keystore", "site": "stackoverflow", "pagesize": 5})
    for item in res.json().get("items", []):
        ans = session.get(f"https://api.stackexchange.com/2.3/questions/{item['question_id']}/answers", params={"site": "stackoverflow", "filter": "withbody"})
        for a in ans.json().get("items", []):
            links = re.findall(r'https?://[^\s"\'<>]+\.jks', a.get("body", ""))
            for l in links:
                candidates.append({"Platform": "StackOverflow", "Repository": "N/A", "Branch": "N/A", "Path": os.path.basename(l), "Repository URL": item["link"], "File URL": l})
except Exception: pass

# ۵. سورس‌فورج (SourceForge HTML Scraping)
print("جستجو در SourceForge...")
try:
    res = session.get("https://sourceforge.net/directory/?q=jks")
    soup = BeautifulSoup(res.text, 'html.parser')
    for a in soup.find_all('a', href=True):
        if '.jks' in a['href']:
            candidates.append({"Platform": "SourceForge", "Repository": "N/A", "Branch": "N/A", "Path": os.path.basename(a['href']), "Repository URL": "https://sourceforge.net", "File URL": a['href']})
except Exception: pass

# حذف تکراری‌ها و ساخت دیتافریم (دقیقاً مثل فایل خودتان)
candidates_df = pd.DataFrame(candidates)
if not candidates_df.empty:
    candidates_df = candidates_df.drop_duplicates(subset=["Platform", "File URL"]).reset_index(drop=True)

print(f"\nفایل‌های JKS پیدا‌شده: {len(candidates_df)}")
display(candidates_df.head())

جستجو در GitHub...
جستجو در GitLab...
جستجو در Bitbucket...
جستجو در StackOverflow...
جستجو در SourceForge...

فایل‌های JKS پیدا‌شده: 34


,Platform,Repository,Branch,Path,Repository URL,File URL
0,GitHub,lenchv/jks-js,master,examples/assets/keystore.jks,https://github.com/lenchv/jks-js,https://github.com/lenchv/jks-js/blob/master/e...
1,GitHub,lenchv/jks-js,master,examples/assets/truststore.jks,https://github.com/lenchv/jks-js,https://github.com/lenchv/jks-js/blob/master/e...
2,GitHub,lenchv/jks-js,master,tests/feature/java11/keystore/DSA_1024_keystor...,https://github.com/lenchv/jks-js,https://github.com/lenchv/jks-js/blob/master/t...
3,GitHub,lenchv/jks-js,master,tests/feature/java11/keystore/EC_256_keystore.jks,https://github.com/lenchv/jks-js,https://github.com/lenchv/jks-js/blob/master/t...
4,GitHub,lenchv/jks-js,master,tests/feature/java11/keystore/RSA_2048_MD5with...,https://github.com/lenchv/jks-js,https://github.com/lenchv/jks-js/blob/master/t...


In [10]:
certificate_rows = []
scan_rows = []

def format_name(name):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        try: return name.rfc4514_string()
        except Exception: return str(name)

for index, item in candidates_df.iterrows():
    platform = item.get("Platform", "Unknown")
    repo_url = item.get("Repository URL", "")
    file_url = item.get("File URL", "")

    # برای گیت‌هاب، لینک Raw می‌سازیم (مثل کد خودتان)
    if platform == "GitHub":
        raw_url = file_url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
    else:
        raw_url = file_url # برای بقیه، لینک مستقیم است

    found_count = 0
    try:
        response = session.get(raw_url, timeout=30)
        response.raise_for_status()
        data = response.content

        ks = jks.KeyStore.loads(data, store_password=None, try_decrypt_keys=False)
        entries = []
        for alias, entry in ks.certs.items(): entries.append((alias, entry.cert, "TrustedCertEntry"))
        for alias, entry in ks.private_keys.items():
            for i, (_, c_data) in enumerate(entry.cert_chain or []): entries.append((f"{alias}_chain_{i}", c_data, "PrivateKeyEntry certificate"))

        for alias, cert_data, entry_type in entries:
            try:
                cert = x509.load_der_x509_certificate(cert_data)
                nb = getattr(cert, "not_valid_before_utc", cert.not_valid_before.replace(tzinfo=timezone.utc))
                na = getattr(cert, "not_valid_after_utc", cert.not_valid_after.replace(tzinfo=timezone.utc))
                now = datetime.now(timezone.utc)
                validity = "Not yet valid" if now < nb else ("Expired" if now > na else "Within date range")

                certificate_rows.append({
                    "Platform": platform, "Repository URL": repo_url, "File URL": file_url,
                    "Alias": alias, "Entry Type": entry_type, "Subject": format_name(cert.subject),
                    "Issuer": format_name(cert.issuer), "Serial Number": format(cert.serial_number, "X"),
                    "Valid From UTC": nb.isoformat(), "Valid Until UTC": na.isoformat(),
                    "Validity Status": validity, "SHA256 Fingerprint": cert.fingerprint(hashes.SHA256()).hex(":").upper(),
                    "Public Key Type": type(cert.public_key()).name, "Signature Algorithm": cert.signature_algorithm_oid.dotted_string,
                    "Integrity Check": "Skipped: no store password"
                })
                found_count += 1
            except Exception as e:
                certificate_rows.append({"Platform": platform, "File URL": file_url, "Alias": alias, "Status": "Certificate parsing failed", "Error": str(e)[:200]})

        scan_rows.append({
            "Platform": platform, "Repository URL": repo_url, "File URL": file_url,
            "File Size (bytes)": len(data), "Entry Count": len(ks.entries), "Certificate Count": found_count,
            "Status": "Parsed; integrity not verified" if found_count else "No certificates processed"
        })
    except Exception as e:
        scan_rows.append({"Platform": platform, "File URL": file_url, "Status": "Unreadable or unsupported JKS", "Error": str(e)[:200]})

certificates_df = pd.DataFrame(certificate_rows)
scan_results_df = pd.DataFrame(scan_rows)
print(f"✅ تعداد گواهی‌های استخراج‌شده: {len(certificates_df)}")

✅ تعداد گواهی‌های استخراج‌شده: 117


In [11]:
output = "/content/jksfinder_analysis.xlsx"
with pd.ExcelWriter(output, engine="openpyxl") as writer:
    certificates_df.to_excel(writer, sheet_name="Certificates", index=False)
    scan_results_df.to_excel(writer, sheet_name="Scan Results", index=False)
print("فایل اکسل آماده است.")

فایل اکسل آماده است.


In [12]:
from google.colab import files

output = "/content/jksfinder_analysis.xlsx"
with pd.ExcelWriter(output, engine="openpyxl") as writer:
    certificates_df.to_excel(writer, sheet_name="Certificates", index=False)
    scan_results_df.to_excel(writer, sheet_name="Scan Results", index=False)

print("✅ فایل اکسل ساخته شد. در حال دانلود...")
files.download(output)

✅ فایل اکسل ساخته شد. در حال دانلود...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import zipfile
import os
from google.colab import files as colab_files

zip_path = "/content/jksfinder_files.zip"
download_dir = "/content/downloaded_jks"
os.makedirs(download_dir, exist_ok=True)

print("⏳ در حال دانلود فایل‌های JKS برای قرار دادن در زیپ...")
for idx, row in candidates_df.iterrows():
    file_url = row.get("File URL", "")
    if row["Platform"] == "GitHub":
        file_url = file_url.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
    try:
        r = session.get(file_url, timeout=30)
        if r.status_code == 200:
            name = f"{row['Platform']}_{idx}_{os.path.basename(row['Path'])}"
            with open(os.path.join(download_dir, name), "wb") as f:
                f.write(r.content)
    except:
        pass

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk(download_dir):
        for filename in filenames:
            zf.write(os.path.join(root, filename), filename)

print(f"✅ فایل زیپ آماده شد: {zip_path}")
colab_files.download(zip_path)

⏳ در حال دانلود فایل‌های JKS برای قرار دادن در زیپ...
✅ فایل زیپ آماده شد: /content/jksfinder_files.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
import os
import zipfile
import re
import pandas as pd
import jks
import glob
from google.colab import files

print("📌 لطفاً فایل اکسل (jksfinder_analysis.xlsx) را از کامپیوتر خود انتخاب کنید:")
uploaded = files.upload()

# پیدا کردن فایل اکسل آپلود شده
excel_filename = None
for fname in uploaded.keys():
    if fname.endswith(".xlsx"):
        excel_filename = fname
        break

if not excel_filename:
    print("❌ فایلی با پسوند xlsx آپلود نشد.")
else:
    print(f"✅ فایل {excel_filename} آپلود شد.")

    # مسیر فایل زیپ که در مرحله قبل ساختیم
    zip_file_path = "/content/jksfinder_files.zip"
    extract_path = "/content/extracted_jks_check"
    os.makedirs(extract_path, exist_ok=True)

    print("⏳ در حال اکسترکت کردن فایل زیپ در کولب...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    print("⏳ در حال استخراج الیاس‌ها از فایل‌های JKS...")
    jks_aliases = []
    for file_path in glob.glob(os.path.join(extract_path, "*.jks")):
        try:
            ks = jks.KeyStore.load(file_path, store_password=None, try_decrypt_keys=False)
            for alias in ks.certs.keys(): jks_aliases.append(alias)
            for alias in ks.private_keys.keys(): jks_aliases.append(alias)
        except: pass

    jks_aliases_set = set(jks_aliases)
    print(f"✅ تعداد {len(jks_aliases_set)} الیاس یکتا از فایل‌های JKS استخراج شد.")

    # خواندن اکسل آپلود شده
    print("⏳ در حال خواندن الیاس‌های اکسل...")
    try:
        excel_df = pd.read_excel(excel_filename, sheet_name="Certificates")

        def clean_alias(a):
            return re.sub(r'_chain_\d+$', '', str(a).strip())

        excel_aliases_set = set(excel_df['Alias'].dropna().apply(clean_alias))

        print("\n" + "="*50)
        print("📊 گزارش تطابق الیاس‌ها (Aliases)")
        print("="*50)
        print(f"تعداد الیاس‌های یکتای اکسل: {len(excel_aliases_set)}")
        print(f"تعداد الیاس‌های یکتای استخراج‌شده از JKS: {len(jks_aliases_set)}")

        missing_in_jks = excel_aliases_set - jks_aliases_set
        extra_in_jks = jks_aliases_set - excel_aliases_set

        if not missing_in_jks and not extra_in_jks:
            print("\n✅ تایید نهایی: تطابق ۱۰۰٪! تمام الیاس‌های اکسل دقیقاً با فایل‌های JKS یکی هستند.")
        else:
            if missing_in_jks:
                print(f"\n⚠️ {len(missing_in_jks)} الیاس در اکسل هست ولی در فایل‌های JKS نیست:")
                for a in list(missing_in_jks)[:10]: print(f"   - {a}")

            if extra_in_jks:
                print(f"\n⚠️ {len(extra_in_jks)} الیاس در JKS هست ولی در اکسل نیست:")
                for a in list(extra_in_jks)[:10]: print(f"   - {a}")
        print("="*50)

    except Exception as e:
        print(f"❌ خطا در خواندن اکسل: {e}")

📌 لطفاً فایل اکسل (jksfinder_analysis.xlsx) را از کامپیوتر خود انتخاب کنید:


Saving jksfinder_analysis.xlsx to jksfinder_analysis (1).xlsx
✅ فایل jksfinder_analysis (1).xlsx آپلود شد.
⏳ در حال اکسترکت کردن فایل زیپ در کولب...
⏳ در حال استخراج الیاس‌ها از فایل‌های JKS...
✅ تعداد 102 الیاس یکتا از فایل‌های JKS استخراج شد.
⏳ در حال خواندن الیاس‌های اکسل...

📊 گزارش تطابق الیاس‌ها (Aliases)
تعداد الیاس‌های یکتای اکسل: 102
تعداد الیاس‌های یکتای استخراج‌شده از JKS: 102

✅ تایید نهایی: تطابق ۱۰۰٪! تمام الیاس‌های اکسل دقیقاً با فایل‌های JKS یکی هستند.
